In [6]:
import numpy as np
import pandas as pd
import os
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')

In [1]:
STUDENT_IDS = "311566_327442_327436"

In [12]:
# Mathematical profitability threshold: E(Profit) = P(1)*10 - P(0)*5 > 0 => P(1) > 1/3
# TOP_FEATURES = [254, 190, 264, 389, 159]
PROFIT_THRESHOLD = 0.33333
MAX_TARGETS = 1000
OPTIMAL_FEATURES_COUNT = 5

In [8]:
x_train_final = np.load('selected_data/x_train_final.npy')
x_test_final = np.load('selected_data/x_test_final.npy')
y_train = np.loadtxt('data/y_train.txt', skiprows=1)

indices = np.load('selected_data/final_feature_indices.npy')

In [13]:
TOP_FEATURES = indices[:OPTIMAL_FEATURES_COUNT]

In [14]:
TOP_FEATURES

array([254, 190, 264, 389, 159])

In [15]:
X_train = x_train_final[:, :OPTIMAL_FEATURES_COUNT]
X_test = x_test_final[:, :OPTIMAL_FEATURES_COUNT]

# FINAL MODEL

## Training

In [16]:
base_svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)

In [17]:
internal_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', base_svm)
])

In [18]:
production_pipeline = CalibratedClassifierCV(internal_pipeline, method='sigmoid', cv=3)

In [19]:
production_pipeline.fit(X_train, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",Pipeline(step...m_state=42))])
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,"[<sklearn.cali...0019817DC9940>, <sk

## Prediction & Profit Optimization (on test set)

In [20]:
test_probabilities = production_pipeline.predict_proba(X_test)[:, 1]

In [21]:
viable_indices = np.where(test_probabilities >= PROFIT_THRESHOLD)[0]

In [23]:
print(f"Number of clients above the profitability threshold ({PROFIT_THRESHOLD:.4f}): {len(viable_indices)}")

Number of clients above the profitability threshold (0.3333): 4108


In [24]:
sorted_viable_indices = viable_indices[np.argsort(test_probabilities[viable_indices])[::-1]]

In [25]:
final_selected_obs = sorted_viable_indices[:MAX_TARGETS]

In [26]:
print(f"Final number of selected clients (project limit): {len(final_selected_obs)}")

Final number of selected clients (project limit): 1000


In [28]:
if len(final_selected_obs) > 0:
    print(f"\nSelected Target Audience Statistics")
    print(f"Max probability : {test_probabilities[final_selected_obs[0]]*100:.2f}%")
    print(f"Min probability : {test_probabilities[final_selected_obs[-1]]*100:.2f}%")
    print(f"Mean probability: {test_probabilities[final_selected_obs].mean()*100:.2f}%\n")


Selected Target Audience Statistics
Max probability : 85.62%
Min probability : 67.16%
Mean probability: 73.81%



### Generating output files

In [29]:
vars_filepath = f"{STUDENT_IDS}_vars.txt"
with open(vars_filepath, "w") as f:
    for feature_idx in TOP_FEATURES:
        f.write(f"{feature_idx}\n")
print(f"[+] Exported {len(TOP_FEATURES)} feature indices to: {vars_filepath}")

[+] Exported 5 feature indices to: 311566_327442_327436_vars.txt


In [30]:
obs_filepath = f"{STUDENT_IDS}_obs.txt"
with open(obs_filepath, "w") as f:
    for obs_idx in final_selected_obs:
        f.write(f"{obs_idx}\n")
print(f"[+] Exported {len(final_selected_obs)} client indices to: {obs_filepath}")

[+] Exported 1000 client indices to: 311566_327442_327436_obs.txt
